This exercise is to replicating the `winsor2` function in Stata (winsor by group, and option of truncation)

In [1]:
import numpy as np
import pandas as pd
df = pd.read_csv('../data/comp_sample.csv', parse_dates=['datadate'])

In [2]:
def winsorize_by_group(
                        df: pd.DataFrame,
                        var: list[str],
                        p: float = 0.01,
                        truncate: bool = False,
                        by: list[str] | None = None
                        ) -> pd.DataFrame:
    """
    Winsorize or truncate the specified variables in a DataFrame by group.
    
    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame containing the data to be winsorized or truncated.
    var : list[str]
        A list of column names in the DataFrame to be winsorized or truncated.
    p : float, optional
        The proportion of data to be winsorized or truncated from each tail (default is 0.01).
    truncate : bool, optional
        If True, the function will truncate the values outside the specified quantiles to NaN.
        If False, the function will winsorize the values by clipping them to the specified quantiles (default is False).
    by : list[str] | None, optional
        A list of column names to group by before applying the winsorization or truncation. If None, the function will apply the operation to the entire DataFrame (default is None).
    """

    if not (0 < p < 0.5):
        raise ValueError("p must be between 0 and 0.5")

    df_out = df.copy()

    for v in var:
        if by is None:
            lower = df_out[v].quantile(p)
            upper = df_out[v].quantile(1 - p)
        else:
            lower = df_out.groupby(by)[v].transform(lambda s: s.quantile(p))
            upper = df_out.groupby(by)[v].transform(lambda s: s.quantile(1 - p))

        if truncate:
            df_out[f"{v}_t"] = df_out[v].where((df_out[v] >= lower) & (df_out[v] <= upper), np.nan)
        else:
            df_out[f"{v}_w"] = df_out[v].clip(lower, upper)

    return df_out

In [3]:
# Check your function with:
df1 = winsorize_by_group(df = df, var=['at', 'ceq', 'ni'], truncate=True, by = ['fyear'])